# Scores Finding (dedupe to best, then top-K)

**Company:** MongoDB (GothamLoop question bank) · **Category:** Coding · **Tags:** Live Screen, Hash Tables, Sorting · **Difficulty/Frequency:** Uncommon (3/10)

## Concepts

**What this problem is really testing:**
- Seeing that this is **two** separate problems — *aggregate*, then *rank* — and that each has its own right tool
- Knowing that **top-K does not need a full sort**
- Distinguishing `n` (input rows) from `m` (unique players) when stating complexity

**First-principles primer — what is each piece?**

- **Aggregation ≠ ordering.** "Keep the best score per player" is a *grouping* operation. Sorting would achieve it, but sorting answers a much harder question (the total order of everything) than the one being asked. A hash map answers exactly this one, in one pass.
- **Top-K with a heap.** To find the K largest out of m items you do **not** need to sort all m. Keep a **min-heap of size K**: push each item; if the heap grows past K, pop the smallest. The heap always holds the K best seen so far, and the smallest of them sits at the top, ready to be evicted. Cost: **O(m log K)** instead of O(m log m).
- **Why the *min*-heap for the *largest* K.** It feels backwards, but the thing you need instant access to is the **weakest survivor** — the one to throw out when a better candidate arrives. That is the min.
- **`n` vs `m`.** `n` is how many rows you read; `m` is how many distinct players there are. A million rows might hold a thousand players. Saying "O(n + m log K)" rather than "O(n log n)" is the whole complexity discussion.

**The two-phase shape:**

| Phase | Question | Tool | Cost |
|---|---|---|---|
| 1 | "What is each player's best?" | hash map | O(n) |
| 2 | "Who are the top 50?" | size-50 heap | O(m log 50) |

Since K = 50 is a **constant**, `log K` is a constant too — so the whole thing is effectively **O(n)**. That is the sentence to say out loud.

**The bug hiding in the hint.** Hint 3 proposes:

```python
best[name] = max(best.get(name, 0), score)
```

If scores can be **negative**, that default of `0` invents a score the player never had. A player whose only entry is `-5` gets recorded as `0` and outranks someone on `-1`. Use `float("-inf")`, or test `name not in best` instead. The notebook asserts this.

**Simple worked example.** `[("ann",10), ("bob",30), ("ann",25), ("cy",5), ("bob",20)]`, top 2:

| step | `best` |
|---|---|
| ann 10 | `{ann: 10}` |
| bob 30 | `{ann: 10, bob: 30}` |
| ann 25 | `{ann: **25**, bob: 30}` — 25 > 10 |
| cy 5 | `{ann: 25, bob: 30, cy: 5}` |
| bob 20 | unchanged — 20 < 30, **keep the max, not the latest** |

Top 2 → `[("bob", 30), ("ann", 25)]`. Note bob's later, *lower* score did not overwrite his best.

## Problem Statement

Given player records `(name, score)` where a player may appear many times:

1. Keep only each player's **highest** score.
2. Return the **top 50**, sorted by score **descending**.

```python
top_players([Player("ann", 10), Player("bob", 30),
             Player("ann", 25), Player("bob", 20)], k=2)
# -> [("bob", 30), ("ann", 25)]
```

In [ ]:
import heapq
import random
from collections import namedtuple
from typing import Dict, Iterable, List, Tuple

Player = namedtuple("Player", ["name", "score"])

### Approach 1 — Naive (sort everything, twice)

**Idea:** sort the whole input by `(name, -score)` so each player's best is first, walk it keeping first occurrences, then sort those by score and slice the top 50.

Correct, and it is what most people write first. It is also doing far more work than asked: two full sorts, when phase 1 needs no ordering at all and phase 2 needs only the top 50 in order.

**Time complexity:** **O(n log n + m log m)** — two full sorts.

**Space complexity:** O(n) for the sorted copy.

In [ ]:
def top_players_naive(players: Iterable[Player], k: int = 50) -> List[Tuple[str, int]]:
    rows = sorted(players, key=lambda p: (p.name, -p.score))   # O(n log n)
    best: List[Tuple[str, int]] = []
    for p in rows:
        if not best or best[-1][0] != p.name:
            best.append((p.name, p.score))       # first of each name = its highest
    best.sort(key=lambda kv: -kv[1])             # O(m log m) - a SECOND full sort
    return best[:k]                              # ...to keep only 50 of them

### Approach 2 — Hash map, then a full sort

**Idea:** fix phase 1. A dict keyed by name collapses duplicates in **one pass** — no ordering needed, because "what is the max for this key?" is a grouping question, not a sorting one.

Phase 2 still sorts all m unique players to take 50, which is the remaining waste.

**Time complexity:** O(n) to aggregate + **O(m log m)** to sort.

**Space complexity:** O(m).

In [ ]:
def top_players_sort(players: Iterable[Player], k: int = 50) -> List[Tuple[str, int]]:
    best: Dict[str, int] = {}
    for p in players:
        # NOT `best.get(name, 0)` - that default invents a 0 for negative-only scores
        if p.name not in best or p.score > best[p.name]:
            best[p.name] = p.score               # O(1) average, one pass
    return sorted(best.items(), key=lambda kv: -kv[1])[:k]     # sorts ALL m to keep k

### Approach 3 — Optimal (hash map, then a size-K heap)

**Idea:** fix phase 2 as well. `heapq.nlargest(k, ...)` keeps a **min-heap of size k** and streams through the m entries: push, and whenever the heap exceeds k, pop the smallest. The heap therefore always holds the k best seen so far, and the item to evict is instantly available at the top.

**O(m log k)** instead of O(m log m). With k = 50 that `log k` is under 6 — effectively a constant — so the whole function is O(n).

**The tie-break.** Two players on the same score have no defined order, so the output is **non-deterministic** between runs. That is a real problem at the boundary of the top 50: whether you make the cut could change run to run. Sorting by `(-score, name)` makes it deterministic, and flagging the ambiguity is worth more than either choice.

**Time complexity:** **O(n + m log k)** ≈ O(n) for constant k.

**Space complexity:** O(m) for the map, O(k) for the heap.

In [ ]:
def top_players(players: Iterable[Player], k: int = 50) -> List[Tuple[str, int]]:
    best: Dict[str, int] = {}
    for p in players:
        if p.name not in best or p.score > best[p.name]:
            best[p.name] = p.score
    # A min-heap of size k: O(m log k), never sorting all m.
    return heapq.nlargest(k, best.items(), key=lambda kv: kv[1])


def top_players_deterministic(players: Iterable[Player], k: int = 50) -> List[Tuple[str, int]]:
    """Same, but ties are broken alphabetically so the output is reproducible."""
    best: Dict[str, int] = {}
    for p in players:
        if p.name not in best or p.score > best[p.name]:
            best[p.name] = p.score
    # nlargest wants the BIGGEST key first, so negate the name's ordering by
    # sorting the shortlist afterwards - k is small, so this costs nothing.
    shortlist = heapq.nlargest(k, best.items(), key=lambda kv: kv[1])
    ties = min((s for _, s in shortlist), default=None)
    if ties is not None:
        # Pull in every entry tied with the cutoff, then order deterministically.
        boundary = [kv for kv in best.items() if kv[1] == ties]
        shortlist = [kv for kv in shortlist if kv[1] != ties] + boundary
    shortlist.sort(key=lambda kv: (-kv[1], kv[0]))
    return shortlist[:k]

### Follow-up — a live leaderboard, where scores can also go *down*

**Idea:** the max-dict silently assumes scores only ever improve. The moment a score can **decrease**, `best[name] = max(...)` is wrong — it would keep a stale high-water mark rather than the current value.

For a live top-K you need two things:

- A **dict** holding each player's *current* score (the source of truth).
- A **heap** for fast ranking — but a heap cannot update an entry in place.

The standard fix is **lazy deletion**: never remove the stale heap entry. Push a new one, and when popping, discard any entry whose score no longer matches the dict. Stale entries accumulate, so the heap is rebuilt once it grows past a threshold — which keeps the amortised cost O(log n) per update.

**Time complexity:** O(log n) amortised per update; O(k log n) per top-K query.

**Space complexity:** O(n + stale entries), bounded by the rebuild threshold.

In [ ]:
class Leaderboard:
    """Live top-K where scores can go up OR down. Lazy deletion keeps the heap honest."""

    def __init__(self) -> None:
        self.scores: Dict[str, int] = {}         # the SOURCE OF TRUTH
        self._heap: List[Tuple[int, str]] = []   # (-score, name); may hold stale entries

    def set_score(self, name: str, score: int) -> None:
        self.scores[name] = score                # overwrite: this is the CURRENT score
        heapq.heappush(self._heap, (-score, name))
        if len(self._heap) > 2 * len(self.scores) + 16:
            self._rebuild()                      # bound the accumulated garbage

    def remove(self, name: str) -> None:
        self.scores.pop(name, None)              # the stale heap entry is left to be skipped

    def _rebuild(self) -> None:
        self._heap = [(-s, n) for n, s in self.scores.items()]
        heapq.heapify(self._heap)                # O(n), cheaper than n pushes

    def top(self, k: int) -> List[Tuple[str, int]]:
        out: List[Tuple[str, int]] = []
        buffer: List[Tuple[int, str]] = []
        while self._heap and len(out) < k:
            neg, name = heapq.heappop(self._heap)
            if self.scores.get(name) != -neg:
                continue                         # STALE: superseded or deleted - drop it
            out.append((name, -neg))
            buffer.append((neg, name))
        for item in buffer:
            heapq.heappush(self._heap, item)     # put the live ones back
        return out

## Verification

The worked example, then the cases that separate a correct implementation from a plausible one: keeping the **max** rather than the latest, negative scores, fewer players than k, and ties.

In [ ]:
IMPLS = [top_players, top_players_sort, top_players_naive]

SAMPLE = [Player("ann", 10), Player("bob", 30), Player("ann", 25),
          Player("cy", 5), Player("bob", 20)]

# --- The worked example ---
for fn in IMPLS:
    assert fn(SAMPLE, k=2) == [("bob", 30), ("ann", 25)], fn.__name__
    assert fn(SAMPLE, k=3) == [("bob", 30), ("ann", 25), ("cy", 5)], fn.__name__

# --- The MAX is kept, not the latest or the first ---
for fn in IMPLS:
    assert fn([Player("x", 5), Player("x", 1)], k=1) == [("x", 5)], f"{fn.__name__}: not the last"
    assert fn([Player("x", 1), Player("x", 5)], k=1) == [("x", 5)], f"{fn.__name__}: not the first"
    assert fn([Player("x", 3), Player("x", 9), Player("x", 3)], k=1) == [("x", 9)], fn.__name__

# --- THE hint's bug: a `0` default invents a score for negative-only players ---
negatives = [Player("a", -5), Player("b", -1), Player("c", -100)]
for fn in IMPLS:
    assert fn(negatives, k=3) == [("b", -1), ("a", -5), ("c", -100)], (
        f"{fn.__name__}: negative scores must survive intact"
    )
    assert fn([Player("solo", -7)], k=1) == [("solo", -7)], (
        f"{fn.__name__}: a lone negative score must NOT become 0"
    )

# What the buggy version would do, for contrast
buggy = {}
for p in negatives:
    buggy[p.name] = max(buggy.get(p.name, 0), p.score)     # the hint's version
assert buggy == {"a": 0, "b": 0, "c": 0}, "the 0 default erases every negative score"

# --- Edge cases ---
for fn in IMPLS:
    assert fn([], k=50) == [], f"{fn.__name__}: empty input"
    assert fn(SAMPLE, k=0) == [], f"{fn.__name__}: k = 0"
    assert len(fn(SAMPLE, k=50)) == 3, f"{fn.__name__}: fewer unique players than k"
    assert fn([Player("only", 1)], k=50) == [("only", 1)], fn.__name__

# The result is always sorted descending
for fn in IMPLS:
    out = fn(SAMPLE, k=50)
    assert out == sorted(out, key=lambda kv: -kv[1]), f"{fn.__name__}: must be descending"

# --- The default k really is 50 ---
many = [Player(f"p{i}", i) for i in range(200)]
assert len(top_players(many)) == 50
assert top_players(many)[0] == ("p199", 199)
assert top_players(many)[-1] == ("p150", 150), "exactly the top 50 by score"

# --- Ties: the multiset of scores must agree even when the order does not ---
tied = [Player(f"p{i}", 7) for i in range(10)] + [Player("high", 99)]
for fn in IMPLS:
    out = fn(tied, k=5)
    assert out[0] == ("high", 99), f"{fn.__name__}: the clear winner is first"
    assert [s for _, s in out] == [99, 7, 7, 7, 7], f"{fn.__name__}: the rest are all tied"

# The deterministic variant breaks ties alphabetically, reproducibly
det = top_players_deterministic(tied, k=5)
assert det == [("high", 99), ("p0", 7), ("p1", 7), ("p2", 7), ("p3", 7)], det
for _ in range(20):
    shuffled = tied[:]
    random.shuffle(shuffled)
    assert top_players_deterministic(shuffled, k=5) == det, (
        "a deterministic tie-break must not depend on input order"
    )

# --- All implementations produce a VALID top-k on randomised data ---
# With ties, two correct implementations may legitimately pick DIFFERENT players,
# so rather than demanding identical output, assert the properties that must hold.
def assert_valid_topk(got, rows, k, who):
    ref: Dict[str, int] = {}                 # independent brute-force reference
    for p in rows:
        ref[p.name] = max(ref[p.name], p.score) if p.name in ref else p.score

    assert len(got) == min(k, len(ref)), (who, "wrong length")
    assert len(set(n for n, _ in got)) == len(got), (who, "a player appears twice")
    for name, score in got:
        assert ref[name] == score, (who, name, "score is not that player's maximum")
    assert [s for _, s in got] == sorted((s for _, s in got), reverse=True),         (who, "not sorted descending")
    # The multiset of returned scores must be exactly THE k highest, ties or not.
    assert [s for _, s in got] == sorted(ref.values(), reverse=True)[:k],         (who, "not the k highest scores")


random.seed(67)
for _ in range(400):
    names = [f"p{i}" for i in range(random.randint(1, 30))]
    rows = [Player(random.choice(names), random.randint(-50, 50))
            for _ in range(random.randint(0, 120))]
    k = random.randint(0, 12)
    for fn in (top_players, top_players_sort, top_players_naive, top_players_deterministic):
        assert_valid_topk(fn(rows, k), rows, k, fn.__name__)

    # The deterministic variant must additionally be stable under shuffling.
    a = top_players_deterministic(rows, k)
    shuffled = rows[:]
    random.shuffle(shuffled)
    assert top_players_deterministic(shuffled, k) == a,         "deterministic output must not depend on input order"

# --- Follow-up: a live leaderboard where scores go DOWN ---
lb = Leaderboard()
for name, score in [("ann", 10), ("bob", 30), ("cy", 20)]:
    lb.set_score(name, score)
assert lb.top(3) == [("bob", 30), ("cy", 20), ("ann", 10)]

lb.set_score("bob", 5)                       # bob LOSES points - a max-dict would miss this
assert lb.top(3) == [("cy", 20), ("ann", 10), ("bob", 5)], (
    "the current score must win, not the historical maximum"
)
assert lb.scores["bob"] == 5

lb.set_score("ann", 100)
assert lb.top(1) == [("ann", 100)]
lb.remove("cy")
assert [n for n, _ in lb.top(5)] == ["ann", "bob"], "a removed player must disappear"
assert lb.top(0) == []

# Repeated updates must not corrupt the heap, and must not grow it without bound
for i in range(300):
    lb.set_score(f"q{i % 20}", random.randint(0, 1000))
top = lb.top(5)
assert top == sorted(top, key=lambda kv: -kv[1]), "still descending after heavy churn"
for name, score in top:
    assert lb.scores[name] == score, "every reported score must be the CURRENT one"
assert len(lb._heap) <= 2 * len(lb.scores) + 16 + 300, "stale entries stay bounded"

print("All assertions passed.")

## Discussion — remaining follow-up directions

- **A live stream of updates.** Implemented above as `Leaderboard`. The key realisation is that a **heap cannot update an entry in place** — there is no "find this element and change its priority" operation without an extra index. **Lazy deletion** sidesteps it: push the new value, leave the stale one, and discard entries on pop whose score no longer matches the dict. Stale entries accumulate, so you rebuild once the heap exceeds a multiple of the live set, which keeps the amortised cost at O(log n).
- **Scores that can decrease.** This is what breaks the simple solution, and it is worth being precise about *why*: `max()` records a **high-water mark**, which is the right answer for "best ever" and the wrong one for "current standing". The two are different questions that look identical on data where scores only rise. Ask which one is meant.
- **Too large for memory.** Phase 1 becomes a **map-reduce** with a combiner: partition rows by `hash(name)` so every row for a player lands on the same worker, take the max per key locally, then merge. Phase 2 is easier than it looks — each worker keeps only its **own** top 50 (a size-50 heap, O(1) memory), and the coordinator merges those partial lists. You never need all m players anywhere at once. That is precisely the [k-way merge](../6.%20K_Way_Merge/6.%20K_Way_Merge.ipynb) pattern.
- **Deterministic ties.** Implemented as `top_players_deterministic`. The subtlety worth naming: the ambiguity only *matters* at the **boundary** of the top k — whether a tied player makes the cut can flip between runs, which turns a leaderboard into a source of support tickets. The `(-score, name)` key costs nothing and removes the whole class of problem.
- **When k stops being small.** The heap is O(m log k) and a sort is O(m log m); they converge as `k → m`. Past roughly `k > m/log m` the sort's better constant factor wins outright, and `sorted(...)[:k]` is simpler code. Knowing that the heap is only a win **because k is small and fixed** is the real answer to this follow-up.

## Empirical complexity check

Both the sort and the heap are dominated by the same O(n) aggregation pass, so the benchmark isolates **phase 2**: ranking `m` already-deduplicated players to take the top 50.

| Growth when the player count doubles | What it means |
|---|---|
| ~2x | both are linear-ish here; the interesting part is the **constant** |

`log m` grows from 10 to 13 across this range while `log k` stays at ~5.6, so the heap's advantage is a steady constant factor rather than a change in shape. That is the honest result — and it is why "top-K without a full sort" is a constant-factor win, not an asymptotic one, unless k is genuinely tiny relative to m.

In [ ]:
import os, sys
_root = os.getcwd()
for _ in range(5):
    if os.path.exists(os.path.join(_root, "bench_utils.py")):
        break
    _root = os.path.dirname(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)
from bench_utils import benchmark

K = 50


def make_players(m):
    """m distinct players, each appearing ~3 times, so n = 3m."""
    rng = random.Random(71)
    return ([Player(f"p{rng.randrange(m)}", rng.randint(-1000, 1000)) for _ in range(3 * m)],)


def run_naive(rows):
    top_players_naive(rows, K)          # two full sorts


def run_sort(rows):
    top_players_sort(rows, K)           # hash map + full sort of m


def run_heap(rows):
    top_players(rows, K)                # hash map + size-50 heap


benchmark(
    {"Approach 1 - sort twice": run_naive,
     "Approach 2 - map + full sort": run_sort,
     "Approach 3 - map + size-50 heap": run_heap},
    make_players,
    sizes=[2000, 4000, 8000, 16000],
    repeats=2,
)

## Patterns learned

- **Read the problem as two problems.** "Dedupe to the best, then rank" is aggregation followed by selection. Each half has its own right tool, and solving both with one sort is what makes the naive version slow.
- **Grouping is a hash map's job, not a sort's.** "What is the max per key?" needs no ordering at all. Reaching for `sorted()` to group is the single most common trap in this family of questions.
- **Top-K does not need a full sort.** A min-heap of size K, streamed over m items: O(m log K). And it is a **min**-heap for the K *largest*, because the thing you need at your fingertips is the weakest survivor — the one to evict.
- **Name your variables in the complexity.** `n` rows, `m` distinct keys, `k` results. "O(n + m log k)" says something real; "O(n log n)" hides the entire design.
- **Never default a maximum to zero.** `max(best.get(k, 0), v)` silently invents a score for negative data. Use `-inf`, or test membership.
- **Flag non-determinism at a boundary.** Unspecified tie-breaking is harmless in the middle of a list and a genuine bug at the cut-off. `(-score, name)` costs nothing and removes it.
- **"Best ever" and "current value" are different questions.** They look identical until a value can decrease — at which point `max()` is quietly answering the wrong one.
- **A heap cannot update in place.** When priorities change, use **lazy deletion**: push the new entry, skip stale ones on pop, and rebuild when the garbage outgrows the live set.